# Chapter 5 - RQ2 Global Robustness

This notebook is the working notebook for Chapter 5 of the thesis. It assembles the evidence for RQ2: how do VQGAN and LlamaGen respond when the entire input image is perturbed by global Gaussian noise?

The chapter studies clean ImageNet images and their noisy variants. It does not cover local image patches, token-space interventions, or dataset shift; those belong to later chapters.


## Notebook Contract

This notebook should produce the Chapter 5 tables, figures, and interpretation notes:

- qualitative global-noise examples for each tokenizer;
- image-space PSNR summaries across noise levels;
- end-to-end reconstruction degradation curves;
- token flip-rate curves against the clean tokenization;
- global token-distribution summaries under noise.

The implementation is based on `vq-global-robustness-analysis.ipynb`, but outputs are kept in a single Chapter 5 bundle next to this notebook. Nothing is written to the LaTeX thesis directory.


## 1. Setup and Paths

The old robustness notebook uses run folders under `/exp/...`, which is the path visible from the Jupyter runtime. These defaults are preserved, but the paths can be overridden through environment variables if the server layout changes.

Expected run layout per model:

- `images/**/0_original.png`
- `images/**/1_recon_clean.png`
- `images/**/2_input_noise_low.png`
- `images/**/3_recon_noise_low.png`
- `images/**/4_input_noise_mid.png`
- `images/**/5_recon_noise_mid.png`
- `images/**/6_input_noise_high.png`
- `images/**/7_recon_noise_high.png`
- `metadata_part_*.jsonl`


In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

try:
    from tqdm.contrib.concurrent import process_map
except Exception:
    process_map = None

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
})

SEED = 0
random.seed(SEED)
np.random.seed(SEED)

MODEL_ORDER = ["vqgan", "llamagen"]
MODEL_LABELS = {"vqgan": "VQGAN", "llamagen": "LlamaGen"}
MODEL_COLORS = {"vqgan": "#B85C38", "llamagen": "#2F6F9F"}

@dataclass(frozen=True)
class ExperimentConfig:
    sigma_low: float = 0.1
    sigma_mid: float = 0.2
    sigma_high: float = 0.5
    codebook_size: int = 16_384
    max_workers: int = min(16, os.cpu_count() or 8)
    chunksize: int = 64

CFG = ExperimentConfig()

LEVELS = [
    ("clean", 0.0, "indices_clean"),
    ("low", CFG.sigma_low, "indices_low"),
    ("mid", CFG.sigma_mid, "indices_mid"),
    ("high", CFG.sigma_high, "indices_high"),
]
NOISY_LEVELS = [item for item in LEVELS if item[0] != "clean"]
LEVEL_ORDER = [level for level, _, _ in LEVELS]
LEVEL_TO_SIGMA = {level: sigma for level, sigma, _ in LEVELS}
LEVEL_TO_INDEX_KEY = {level: key for level, _, key in LEVELS}


def find_notebook_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "notebooks", cwd.parent / "notebooks"]
    for candidate in candidates:
        if (candidate / "data").exists() or candidate.name == "notebooks":
            return candidate
    return cwd


NOTEBOOK_DIR = find_notebook_root()
REPO_ROOT = NOTEBOOK_DIR.parent

RUN_DIRS = {
    "vqgan": Path(os.environ.get("RQ2_VQGAN_DIR", "/exp/1765562245_robustness_dataset_vqgan")),
    "llamagen": Path(os.environ.get("RQ2_LLAMAGEN_DIR", "/exp/1765559700_robustness_dataset_llamagen")),
}

OUTPUT_DIR = NOTEBOOK_DIR / "chapter5_outputs"
DATA_DIR = OUTPUT_DIR / "data"
FIGURE_DIR = OUTPUT_DIR / "figures"
SAMPLE_EXPORT_DIR = OUTPUT_DIR / "qualitative_samples"
CACHE_DIR = OUTPUT_DIR / "cache"

for directory in [DATA_DIR, FIGURE_DIR, SAMPLE_EXPORT_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Notebook dir:", NOTEBOOK_DIR)
for model, run_dir in RUN_DIRS.items():
    print(f"{MODEL_LABELS[model]} run dir:", run_dir, "exists:", run_dir.exists())
print("Output dir:", OUTPUT_DIR)


## 2. Loading and Export Helpers

The helpers below are small on purpose: they load the run layout, compute basic image metrics, and save only into `chapter5_outputs/`.


In [ ]:
def save_table(df: pd.DataFrame, name: str) -> Path:
    path = DATA_DIR / name
    df.to_csv(path, index=False)
    print(f"saved: {path}")
    return path


def save_figure(fig: plt.Figure, name: str) -> Path:
    path = FIGURE_DIR / name
    fig.savefig(path, bbox_inches="tight")
    print(f"saved: {path}")
    return path


def save_json(data: dict, name: str) -> Path:
    path = DATA_DIR / name
    path.write_text(json.dumps(data, indent=2) + "\n")
    print(f"saved: {path}")
    return path


def save_df_cache(df: pd.DataFrame, path: Path) -> None:
    try:
        df.to_parquet(path, index=False)
        print(f"cached: {path}")
    except Exception:
        csv_path = path.with_suffix(".csv")
        df.to_csv(csv_path, index=False)
        print(f"cached: {csv_path}")


def load_df_cache(path: Path) -> pd.DataFrame:
    if path.exists():
        return pd.read_parquet(path)
    csv_path = path.with_suffix(".csv")
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f"No cache found at {path} or {csv_path}")


def find_sample_dirs(images_root: Path) -> list[Path]:
    if not images_root.exists():
        raise FileNotFoundError(f"Missing images directory: {images_root}")
    sample_dirs = sorted({p.parent for p in images_root.rglob("0_original.png")})
    if not sample_dirs:
        raise RuntimeError(f"No samples found under {images_root}; expected files like 0_original.png")
    return sample_dirs


def find_metadata_files(run_dir: Path) -> list[Path]:
    files = sorted(run_dir.glob("metadata_part_*.jsonl"))
    if not files:
        raise FileNotFoundError(f"No metadata shards found under {run_dir}")
    return files


def peek_first_jsonl(path: Path) -> dict:
    with path.open("r") as f:
        for line in f:
            if line.strip():
                return json.loads(line)
    raise RuntimeError(f"Empty metadata file: {path}")


def load_img_float01(path: Path) -> np.ndarray:
    with Image.open(path) as im:
        return np.asarray(im.convert("RGB"), dtype=np.float32) / 255.0


def mse(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.mean((a.astype(np.float32) - b.astype(np.float32)) ** 2))


def psnr(a: np.ndarray, b: np.ndarray, max_i: float = 1.0) -> float:
    value = mse(a, b)
    if value == 0.0:
        return float("inf")
    return 10.0 * math.log10((max_i ** 2) / value)


def copy_qualitative_sample(sample_dir: Path, model: str) -> Path:
    destination = SAMPLE_EXPORT_DIR / model / sample_dir.name
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    for path in sorted(sample_dir.glob("*.png")):
        shutil.copy2(path, destination / path.name)
    return destination


## 3. Inventory and Validation

Before computing metrics, the notebook checks that both model runs contain image samples and metadata shards. This protects the later figures from silently mixing mismatched or partial runs.


In [ ]:
runs = {}
for model, run_dir in RUN_DIRS.items():
    images_root = run_dir / "images"
    sample_dirs = find_sample_dirs(images_root)
    metadata_files = find_metadata_files(run_dir)
    first_record = peek_first_jsonl(metadata_files[0])
    runs[model] = {
        "run_dir": run_dir,
        "images_root": images_root,
        "sample_dirs": sample_dirs,
        "metadata_files": metadata_files,
        "first_record": first_record,
    }

inventory_rows = []
for model, run in runs.items():
    inventory_rows.append({
        "model": MODEL_LABELS[model],
        "run_dir": str(run["run_dir"]),
        "n_samples": len(run["sample_dirs"]),
        "n_metadata_shards": len(run["metadata_files"]),
        "first_sample": run["sample_dirs"][0].name,
        "metadata_keys": ", ".join(sorted(run["first_record"].keys())),
    })

inventory_df = pd.DataFrame(inventory_rows)
save_table(inventory_df, "ch05_table_run_inventory.csv")
display(inventory_df)


## 4. Qualitative Sanity Check

The qualitative panel shows the original image, globally perturbed inputs, the clean reconstruction, and reconstructions after encoding noisy inputs. The purpose is not to cherry-pick final evidence, but to verify that the perturbation pipeline is visually plausible.


In [ ]:
PANEL_INPUTS = [
    ("0_original.png", "x"),
    ("2_input_noise_low.png", "x' low"),
    ("4_input_noise_mid.png", "x' mid"),
    ("6_input_noise_high.png", "x' high"),
]
PANEL_RECONS = [
    ("1_recon_clean.png", "D(E(x))"),
    ("3_recon_noise_low.png", "recon low"),
    ("5_recon_noise_mid.png", "recon mid"),
    ("7_recon_noise_high.png", "recon high"),
]


def pick_sample_dirs(sample_dirs: list[Path], n: int, seed: int = SEED) -> list[Path]:
    n = min(n, len(sample_dirs))
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(sample_dirs), size=n, replace=False)
    return [sample_dirs[i] for i in indices]


def plot_noise_grid(model: str, selected_samples: list[Path], max_samples: int = 3) -> plt.Figure:
    selected_samples = selected_samples[:max_samples]
    cols = 4
    rows = 2 * len(selected_samples)
    fig, axes = plt.subplots(rows, cols, figsize=(2.3 * cols, 2.15 * rows), squeeze=False)

    for i, sample_dir in enumerate(selected_samples):
        copy_qualitative_sample(sample_dir, model)
        for col, (filename, label) in enumerate(PANEL_INPUTS):
            ax = axes[2 * i, col]
            ax.imshow(load_img_float01(sample_dir / filename))
            ax.set_title(label if i == 0 else "", fontsize=9)
            ax.set_ylabel(sample_dir.name, fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.grid(False)
        for col, (filename, label) in enumerate(PANEL_RECONS):
            ax = axes[2 * i + 1, col]
            ax.imshow(load_img_float01(sample_dir / filename))
            ax.set_title(label if i == 0 else "", fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.grid(False)

    fig.suptitle(f"{MODEL_LABELS[model]} global-noise qualitative examples", fontsize=13, fontweight="bold")
    return fig


selected_by_model = {model: pick_sample_dirs(run["sample_dirs"], n=4, seed=SEED) for model, run in runs.items()}
for model in MODEL_ORDER:
    fig = plot_noise_grid(model, selected_by_model[model], max_samples=3)
    save_figure(fig, f"ch05_fig_global_noise_qualitative_{model}.png")
    plt.show()


## 5. Image-Space Robustness Metrics

For each sample and noise level, the notebook computes three PSNR quantities:

- `psnr_x_xnoisy`: how strong the input corruption is, measured between clean image `x` and noisy image `x'`;
- `psnr_xnoisy_xhat`: how closely the reconstruction follows the noisy input;
- `psnr_x_xhat`: the end-to-end fidelity of the reconstruction to the original clean image.

The clean baseline is `psnr_x_xcleanrecon = PSNR(x, D(E(x)))`.


In [ ]:
LEVEL_TO_FILES = {
    "clean": ("0_original.png", "1_recon_clean.png", None),
    "low": ("0_original.png", "3_recon_noise_low.png", "2_input_noise_low.png"),
    "mid": ("0_original.png", "5_recon_noise_mid.png", "4_input_noise_mid.png"),
    "high": ("0_original.png", "7_recon_noise_high.png", "6_input_noise_high.png"),
}


def compute_psnr_rows_for_sample(args) -> list[dict]:
    sample_dir, model = args
    x = load_img_float01(sample_dir / "0_original.png")
    clean_recon = load_img_float01(sample_dir / "1_recon_clean.png")
    rows = [{
        "model": model,
        "sample_id": sample_dir.name,
        "level": "clean",
        "sigma": 0.0,
        "psnr_x_xnoisy": np.nan,
        "psnr_xnoisy_xhat": np.nan,
        "psnr_x_xhat": psnr(x, clean_recon),
        "psnr_x_xcleanrecon": psnr(x, clean_recon),
    }]

    for level, sigma, noisy_filename, recon_filename in [
        ("low", CFG.sigma_low, "2_input_noise_low.png", "3_recon_noise_low.png"),
        ("mid", CFG.sigma_mid, "4_input_noise_mid.png", "5_recon_noise_mid.png"),
        ("high", CFG.sigma_high, "6_input_noise_high.png", "7_recon_noise_high.png"),
    ]:
        x_noisy = load_img_float01(sample_dir / noisy_filename)
        x_hat = load_img_float01(sample_dir / recon_filename)
        rows.append({
            "model": model,
            "sample_id": sample_dir.name,
            "level": level,
            "sigma": float(sigma),
            "psnr_x_xnoisy": psnr(x, x_noisy),
            "psnr_xnoisy_xhat": psnr(x_noisy, x_hat),
            "psnr_x_xhat": psnr(x, x_hat),
            "psnr_x_xcleanrecon": np.nan,
        })
    return rows


def psnr_cache_path(model: str) -> Path:
    tag = f"sL{CFG.sigma_low}_sM{CFG.sigma_mid}_sH{CFG.sigma_high}"
    return CACHE_DIR / f"ch05_psnr_rows_{model}_{tag}.parquet"


def compute_psnr_df(model: str, sample_dirs: list[Path], force: bool = False) -> pd.DataFrame:
    cache_path = psnr_cache_path(model)
    if not force and (cache_path.exists() or cache_path.with_suffix(".csv").exists()):
        print(f"loading cached PSNR rows for {model}")
        return load_df_cache(cache_path)

    tasks = [(sample_dir, model) for sample_dir in sample_dirs]
    if process_map is None:
        iterator = tasks if tqdm is None else tqdm(tasks, desc=f"PSNR {model}")
        nested = [compute_psnr_rows_for_sample(task) for task in iterator]
    else:
        nested = process_map(
            compute_psnr_rows_for_sample,
            tasks,
            max_workers=CFG.max_workers,
            chunksize=CFG.chunksize,
            desc=f"PSNR {model}",
        )
    rows = [row for sub in nested for row in sub]
    df = pd.DataFrame(rows).sort_values(["model", "sample_id", "sigma"]).reset_index(drop=True)
    save_df_cache(df, cache_path)
    return df


psnr_rows = []
for model in MODEL_ORDER:
    psnr_rows.append(compute_psnr_df(model, runs[model]["sample_dirs"], force=False))
psnr_df = pd.concat(psnr_rows, ignore_index=True)
save_table(psnr_df, "ch05_psnr_rows.csv")
display(psnr_df.head())


In [ ]:
PSNR_METRICS = ["psnr_x_xnoisy", "psnr_xnoisy_xhat", "psnr_x_xhat", "psnr_x_xcleanrecon"]


def summarize_psnr(df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        df.groupby(["model", "level", "sigma"], sort=True)[PSNR_METRICS]
        .agg(["mean", "std", "median", "min", "max"])
        .reset_index()
    )
    summary.columns = [
        col if isinstance(col, str) else (col[0] if col[1] == "" else f"{col[0]}_{col[1]}")
        for col in summary.columns
    ]
    return summary.sort_values(["model", "sigma", "level"]).reset_index(drop=True)


psnr_summary = summarize_psnr(psnr_df)
save_table(psnr_summary, "ch05_table_psnr_summary.csv")
display(psnr_summary)


## 6. Image-Space Result Figures

The main Chapter 5 image-space figure should show how end-to-end fidelity to the clean image changes as input noise increases. The clean reconstruction baseline is shown as a dashed horizontal line for each model.


In [ ]:
def mean_std_by_model_sigma(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    return (
        df[["model", "sigma", metric]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .groupby(["model", "sigma"], sort=True)[metric]
        .agg(["mean", "std"])
        .reset_index()
    )


def clean_baseline_by_model(df: pd.DataFrame) -> dict[str, float]:
    clean = df[df["level"] == "clean"]
    return clean.groupby("model")["psnr_x_xcleanrecon"].mean().to_dict()


def plot_psnr_metric(metric: str, title: str, ylabel: str = "PSNR (dB)", show_clean_baseline: bool = False) -> plt.Figure:
    stats = mean_std_by_model_sigma(psnr_df, metric)
    baseline = clean_baseline_by_model(psnr_df) if show_clean_baseline else {}
    fig, ax = plt.subplots(figsize=(7.5, 4.8))

    for model in MODEL_ORDER:
        sub = stats[stats["model"] == model].sort_values("sigma")
        ax.errorbar(
            sub["sigma"],
            sub["mean"],
            yerr=sub["std"],
            marker="o",
            linewidth=2,
            capsize=4,
            color=MODEL_COLORS[model],
            label=MODEL_LABELS[model],
        )
        if model in baseline:
            ax.axhline(baseline[model], linestyle="--", linewidth=1.2, color=MODEL_COLORS[model], alpha=0.75)

    ax.set_title(title)
    ax.set_xlabel("Gaussian noise standard deviation")
    ax.set_ylabel(ylabel)
    ax.set_xticks([sigma for _, sigma, _ in LEVELS])
    ax.legend(frameon=False)
    ax.grid(axis="both")
    return fig


fig = plot_psnr_metric(
    "psnr_x_xhat",
    "End-to-end reconstruction fidelity under global noise",
    show_clean_baseline=True,
)
save_figure(fig, "ch05_fig_psnr_x_xhat_vs_sigma.png")
plt.show()

fig = plot_psnr_metric(
    "psnr_xnoisy_xhat",
    "Reconstruction fidelity to the noisy input",
    show_clean_baseline=False,
)
save_figure(fig, "ch05_fig_psnr_xnoisy_xhat_vs_sigma.png")
plt.show()


## 7. Token Flip Robustness

Token flip rate measures the fraction of token positions whose code index changes relative to the clean image tokenization. This directly probes whether small global image perturbations induce large discrete changes.


In [ ]:
def token_flip_cache_path(model: str) -> Path:
    return CACHE_DIR / f"ch05_token_flip_{model}.csv"


def token_flip_table(metadata_files: list[Path], model: str, force: bool = False) -> pd.DataFrame:
    cache_path = token_flip_cache_path(model)
    if not force and cache_path.exists():
        print(f"loading cached token flip table for {model}")
        return pd.read_csv(cache_path)

    rows = []
    iterator = metadata_files if tqdm is None else tqdm(metadata_files, desc=f"token flip {model}")
    for path in iterator:
        with path.open("r") as f:
            for line in f:
                if not line.strip():
                    continue
                record = json.loads(line)
                clean = np.asarray(record["indices_clean"], dtype=np.int32)
                for level, sigma, key in NOISY_LEVELS:
                    noisy = np.asarray(record[key], dtype=np.int32)
                    rows.append({
                        "model": model,
                        "sample_id": record.get("image_id"),
                        "level": level,
                        "sigma": sigma,
                        "token_flip_frac": float((noisy != clean).mean()),
                    })

    df = pd.DataFrame(rows)
    df.to_csv(cache_path, index=False)
    print(f"cached: {cache_path}")
    return df


flip_df = pd.concat(
    [token_flip_table(runs[model]["metadata_files"], model, force=False) for model in MODEL_ORDER],
    ignore_index=True,
)
save_table(flip_df, "ch05_token_flip_rows.csv")
flip_summary = (
    flip_df.groupby(["model", "level", "sigma"], sort=True)["token_flip_frac"]
    .agg(["mean", "std", "median", "min", "max"])
    .reset_index()
)
save_table(flip_summary, "ch05_table_token_flip_summary.csv")
display(flip_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8))
for model in MODEL_ORDER:
    sub = flip_summary[flip_summary["model"] == model].sort_values("sigma")
    ax.errorbar(
        sub["sigma"],
        sub["mean"],
        yerr=sub["std"],
        marker="o",
        linewidth=2,
        capsize=4,
        color=MODEL_COLORS[model],
        label=MODEL_LABELS[model],
    )
ax.set_title("Token flip rate under global noise")
ax.set_xlabel("Gaussian noise standard deviation")
ax.set_ylabel("Fraction of token positions changed")
ax.set_ylim(0, 1)
ax.legend(frameon=False)
ax.grid(axis="both")
save_figure(fig, "ch05_fig_token_flip_vs_sigma.png")
plt.show()


## 8. Global Token-Distribution Shift

The flip-rate metric is position-wise: it asks how many token positions change. The global distribution view asks whether noise changes the aggregate codebook usage distribution: active code count, entropy, perplexity, and top-k concentration.


In [ ]:
def token_counts_cache_path(model: str) -> Path:
    return CACHE_DIR / f"ch05_token_counts_{model}.npz"


def token_counts_from_metadata(metadata_files: list[Path], model: str, force: bool = False) -> dict[str, np.ndarray]:
    cache_path = token_counts_cache_path(model)
    if not force and cache_path.exists():
        data = np.load(cache_path, allow_pickle=True)
        return {key: data[key] for key in data.files}

    counts = {level: np.zeros(CFG.codebook_size, dtype=np.int64) for level, _, _ in LEVELS}
    iterator = metadata_files if tqdm is None else tqdm(metadata_files, desc=f"token counts {model}")
    for path in iterator:
        with path.open("r") as f:
            for line in f:
                if not line.strip():
                    continue
                record = json.loads(line)
                for level, _, key in LEVELS:
                    indices = np.asarray(record[key], dtype=np.int32).ravel()
                    counts[level] += np.bincount(indices, minlength=CFG.codebook_size)

    np.savez_compressed(cache_path, **counts)
    print(f"cached: {cache_path}")
    return counts


def distribution_stats(counts: np.ndarray) -> dict[str, float]:
    total = float(counts.sum())
    if total <= 0:
        raise ValueError("Cannot summarize an empty count vector")
    probs = counts.astype(np.float64) / total
    probs_nonzero = probs[probs > 0]
    entropy = float(-(probs_nonzero * np.log(probs_nonzero)).sum())
    sorted_probs = np.sort(probs)[::-1]
    return {
        "total_tokens": int(total),
        "active_codes": int((counts > 0).sum()),
        "dead_codes": int((counts == 0).sum()),
        "active_fraction": float((counts > 0).mean()),
        "entropy_nats": entropy,
        "perplexity": float(np.exp(entropy)),
        "top_10_mass": float(sorted_probs[:10].sum()),
        "top_100_mass": float(sorted_probs[:100].sum()),
        "top_1000_mass": float(sorted_probs[:1000].sum()),
    }


token_counts = {model: token_counts_from_metadata(runs[model]["metadata_files"], model, force=False) for model in MODEL_ORDER}
rows = []
for model in MODEL_ORDER:
    for level, sigma, _ in LEVELS:
        row = {"model": model, "level": level, "sigma": sigma}
        row.update(distribution_stats(token_counts[model][level]))
        rows.append(row)

distribution_df = pd.DataFrame(rows).sort_values(["model", "sigma"]).reset_index(drop=True)
save_table(distribution_df, "ch05_table_token_distribution_summary.csv")
display(distribution_df)


In [ ]:
def plot_distribution_metric(metric: str, title: str, ylabel: str) -> plt.Figure:
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    for model in MODEL_ORDER:
        sub = distribution_df[distribution_df["model"] == model].sort_values("sigma")
        ax.plot(
            sub["sigma"],
            sub[metric],
            marker="o",
            linewidth=2,
            color=MODEL_COLORS[model],
            label=MODEL_LABELS[model],
        )
    ax.set_title(title)
    ax.set_xlabel("Gaussian noise standard deviation")
    ax.set_ylabel(ylabel)
    ax.legend(frameon=False)
    ax.grid(axis="both")
    return fig


fig = plot_distribution_metric("active_codes", "Active code count under global noise", "Active codes")
save_figure(fig, "ch05_fig_active_codes_vs_sigma.png")
plt.show()

fig = plot_distribution_metric("perplexity", "Effective codebook perplexity under global noise", "Perplexity")
save_figure(fig, "ch05_fig_perplexity_vs_sigma.png")
plt.show()

fig = plot_distribution_metric("top_100_mass", "Top-100 token mass under global noise", "Fraction of assignments")
save_figure(fig, "ch05_fig_top100_mass_vs_sigma.png")
plt.show()


## 9. Joint Chapter 5 Summary

This compact table joins image-space degradation, token flip rate, and distribution statistics at each noise level. It is useful for drafting the Chapter 5 results section.


In [ ]:
image_summary = psnr_summary[[
    "model", "level", "sigma",
    "psnr_x_xhat_mean", "psnr_x_xhat_std",
    "psnr_xnoisy_xhat_mean", "psnr_xnoisy_xhat_std",
]].copy()

flip_compact = flip_summary[["model", "level", "sigma", "mean", "std"]].rename(columns={
    "mean": "token_flip_frac_mean",
    "std": "token_flip_frac_std",
})

joint_summary = (
    image_summary
    .merge(flip_compact, on=["model", "level", "sigma"], how="left")
    .merge(distribution_df, on=["model", "level", "sigma"], how="left")
    .sort_values(["model", "sigma"])
    .reset_index(drop=True)
)

save_table(joint_summary, "ch05_table_joint_global_robustness_summary.csv")
display(joint_summary)


## 10. Thesis Notes Draft

Use these prompts while writing Chapter 5:

- Does image-space reconstruction degrade smoothly with sigma, or does one tokenizer show a sharper collapse?
- Does token flip rate increase proportionally to pixel-level corruption, or does the discrete representation amplify small perturbations?
- Does global noise activate many previously unused codes, concentrate usage into fewer codes, or leave the global distribution mostly stable?
- Are image-space metrics and token-space metrics telling the same story?

Keep the causal language careful: this chapter observes robustness under global perturbation; it does not yet isolate encoder locality or decoder geometry.


## 11. Artifact Checklist

After a clean server run, all outputs should be under `chapter5_outputs/` next to this notebook.

Expected data files:

- `data/ch05_table_run_inventory.csv`
- `data/ch05_psnr_rows.csv`
- `data/ch05_table_psnr_summary.csv`
- `data/ch05_token_flip_rows.csv`
- `data/ch05_table_token_flip_summary.csv`
- `data/ch05_table_token_distribution_summary.csv`
- `data/ch05_table_joint_global_robustness_summary.csv`

Expected figures:

- `figures/ch05_fig_global_noise_qualitative_vqgan.png`
- `figures/ch05_fig_global_noise_qualitative_llamagen.png`
- `figures/ch05_fig_psnr_x_xhat_vs_sigma.png`
- `figures/ch05_fig_psnr_xnoisy_xhat_vs_sigma.png`
- `figures/ch05_fig_token_flip_vs_sigma.png`
- `figures/ch05_fig_active_codes_vs_sigma.png`
- `figures/ch05_fig_perplexity_vs_sigma.png`
- `figures/ch05_fig_top100_mass_vs_sigma.png`

Expected sample copies:

- `qualitative_samples/vqgan/sample_*/...png`
- `qualitative_samples/llamagen/sample_*/...png`

Nothing in this notebook writes into the thesis directory. Final selected artifacts can be copied into the local LaTeX thesis manually after review.
